**`parcel_zonal_stats`**

Run zonal statistics on parcel geometries for a given raster, looping over admin3 municipalities within one or more admin2 departments.

# Configure

In [ ]:
import argparse
import time
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import rasterio.windows
from rasterio.features import rasterize
from shapely.geometry import box
from rasterstats import zonal_stats

import openplaces as op
from openplaces.flow import convert_to_script, test_script
from openplaces.geo.raster import zonal_stats_with_exactextract
from openplaces.recipe import get_output_path, get_recipe_by_id

warnings.filterwarnings('ignore')

In [ ]:
parser = argparse.ArgumentParser(
    description='Run zonal statistics on parcel geometries for a given raster'
)
parser.add_argument(
    '--admin1_id',
    help='Admin1 ID (e.g., "CO")',
)
parser.add_argument(
    '--admin2_id',
    nargs='*',
    help='Admin2 ID(s) to process (e.g., "CO-AN").',
)
parser.add_argument(
    '--raster_path',
    required=True,
    help='Path to input raster file',
)
parser.add_argument(
    '--raster_key',
    required=True,
    help='Label used in output filenames and output column prefix (e.g., "elevation")',
)
parser.add_argument(
    '--stat',
    choices=['mean', 'max', 'min', 'sum', 'std'],
    help='Aggregation statistic',
)
parser.add_argument(
    '--method',
    choices=['exactextract', 'rasterstats', 'rasterized'],
    help='Zonal statistics method',
)
parser.add_argument(
    '--recipe_id',
    nargs='*',
    help=(
        'Parcel recipe id. Can accept a single parcel recipe or multiple.'
        'Example (pre harmonization): "CO_parcel-igac-2026_rural,CO_parcel-igac-2026_urban" "CO-AN_parcel"'
    ),
)
parser.add_argument(
    '--reprocess',
    action='store_true',
    help='Rerun even if a checkpoint already exists',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--admin1_id CO '
    '--admin2_id CO-CA '
    '--raster_path /Users/alexanderweintraub/openplaces/data/cache/CO/_all/DEM/copernicus_30m/CO_DEM_copernicus_30m.tif '
    '--raster_key elevation '
    '--stat max '
    '--method exactextract '
    '--recipe_id CO_parcel-igac-2026_rural,CO_parcel-igac-2026_urban CO-AN_parcel '
    # '--reprocess '
)

args_list = [x for x in ARGS_TEST.split(' ') if x != '']
args = parser.parse_args(args_list)
args

# Setup

In [ ]:
COUNTRY = args.admin1_id
ADMIN2_IDS = args.admin2_id
RASTER_KEY = args.raster_key
STAT = args.stat
METHOD = args.method
RASTER_PATH = Path(args.raster_path)

# Recipe groups: tried in priority order per admin3 unit.
# This step will become unessesary once harmonize step can merge parcels from multiple recipes
PARCEL_RECIPE_GROUPS = [g.split(',') for g in (args.recipe_groups or [])]
CACHED_RECIPES = {
    rid: get_recipe_by_id(rid) for group in PARCEL_RECIPE_GROUPS for rid in group
}

with rasterio.open(RASTER_PATH) as src:
    RASTER_CRS = src.crs
    RASTER_BOUNDS = src.bounds
    print(
        f'Raster: {src.width}x{src.height}  CRS={src.crs.to_epsg()}  nodata={src.nodata}'
    )

RASTER_EXTENT = box(*RASTER_BOUNDS)
COL_NAME = f'{RASTER_KEY}_{STAT}'
UNIQUE_ID = 'parcel_id'

print(f'Raster key: {RASTER_KEY}')
print(f'Stat:       {STAT}  ->  column: {COL_NAME}')
print(f'Method:     {METHOD}')
print(f'Recipe groups: {PARCEL_RECIPE_GROUPS}')

# Helper functions

In [ ]:
def find_recipe_group(admin3_id):
    """Return (recipe_ids, primary_path) for the first group with data on disk.
    This function allows the user to process parcels from different sources,
    and identifies which recipe source should be used for each admin3"""
    for group in PARCEL_RECIPE_GROUPS:
        for rid in group:
            try:
                p = get_output_path(CACHED_RECIPES[rid], admin3_id)
                if p.exists():
                    primary = get_output_path(CACHED_RECIPES[group[0]], admin3_id)
                    return group, primary
            except Exception:
                continue
    return None, None


def load_parcels(admin3_id, recipe_ids):
    """Load and concat parcels from one or more recipes for a single admin3.
    This function allows the user to combine parcels from different recipe sources"""
    frames = []
    for rid in recipe_ids:
        try:
            gdf = op.get_entities(rid, admin3_id, geom=True)
            frames.append(gdf)
        except Exception:
            pass
    if not frames:
        return None
    combined = gpd.GeoDataFrame(
        pd.concat(frames), geometry='geometry', crs=frames[0].crs
    )
    combined = combined[combined.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
    return combined if not combined.empty else None


# Note that find_recipe_group and load_parcels are temporary functions that will converted for the harmonize step


def stats_exactextract(parcels_r, raster_path):
    result_gdf = zonal_stats_with_exactextract(
        parcels_r,
        raster_path,
        stats=[STAT],
        col_prefix=f'{RASTER_KEY}_',
        reproject=False,
        clean_geometry=False,
    )
    return result_gdf[[COL_NAME]]


def stats_rasterstats(parcels_r, raster_path):
    raw = zonal_stats(
        vectors=parcels_r.geometry,
        raster=str(raster_path),
        stats=[STAT],
        all_touched=False,
        nodata=rasterio.open(raster_path).nodata,
    )
    result = pd.DataFrame(raw, index=parcels_r.index)[[STAT]]
    return result.rename(columns={STAT: COL_NAME})


def stats_rasterized(parcels_r, raster_path):
    """Burn polygons to raster grid, then aggregate with numpy groupby."""
    STAT_FN = {
        'mean': np.nanmean,
        'max': np.nanmax,
        'min': np.nanmin,
        'sum': np.nansum,
        'std': np.nanstd,
        'count': lambda x: np.sum(~np.isnan(x)),
    }
    if STAT not in STAT_FN:
        raise ValueError(
            f'rasterized method does not support stat={STAT!r}. '
            f'Choose from: {list(STAT_FN)}'
        )
    agg_fn = STAT_FN[STAT]

    with rasterio.open(raster_path) as src:
        bounds = parcels_r.total_bounds
        win = src.window(*bounds)
        win = win.intersection(rasterio.windows.Window(0, 0, src.width, src.height))
        win_transform = src.window_transform(win)
        raster_data = src.read(1, window=win)
        nodata = src.nodata

    parcel_ids = parcels_r.index.tolist()
    int_indices = range(1, len(parcel_ids) + 1)
    idx_to_pid = dict(zip(int_indices, parcel_ids))

    shapes = ((geom, idx) for geom, idx in zip(parcels_r.geometry, int_indices))
    id_grid = rasterize(
        shapes,
        out_shape=raster_data.shape,
        transform=win_transform,
        fill=0,
        dtype='int32',
    )

    raster_f = raster_data.astype('float32')
    if nodata is not None:
        raster_f[raster_data == nodata] = np.nan

    flat_ids = id_grid.ravel()
    flat_vals = raster_f.ravel()
    mask = flat_ids > 0

    df = pd.DataFrame({'idx': flat_ids[mask], 'val': flat_vals[mask]})
    grouped = df.groupby('idx')['val'].agg(agg_fn)
    result_vals = {idx_to_pid[i]: v for i, v in grouped.items() if i in idx_to_pid}
    result = pd.Series(result_vals, name=COL_NAME).reindex(parcel_ids)
    result.index.name = UNIQUE_ID
    return result.to_frame()


METHOD_FN = {
    'exactextract': stats_exactextract,
    'rasterstats': stats_rasterstats,
    'rasterized': stats_rasterized,
}


def run_zonal_stats(parcels, raster_path):
    """Reproject, spatial-filter, then call the selected method."""
    parcels_r = parcels.to_crs(RASTER_CRS)
    parcels_r = parcels_r[parcels_r.intersects(RASTER_EXTENT)]
    if parcels_r.empty:
        return None
    return METHOD_FN[METHOD](parcels_r, raster_path)


def get_output_stats_path(admin3_id, recipe_ids):
    """Stats parquet lives next to the primary parcel parquet."""
    p = get_output_path(CACHED_RECIPES[recipe_ids[0]], admin3_id)
    return p.parent / f'{p.stem}_{RASTER_KEY}_{STAT}.parquet'

# Run zonal statistics

In [ ]:
for admin2_id in ADMIN2_IDS:
    admin3_ids = op.get_admin_ids(admin_level=3, admin_id=admin2_id)
    print(f'\n{len(admin3_ids)} municipalities in {admin2_id}  [method={METHOD}]')

    for admin3_id in admin3_ids:
        recipe_ids, _ = find_recipe_group(admin3_id)
        if recipe_ids is None:
            print(f'[{admin3_id}] SKIP — no parcel files found')
            continue

        ckpt = get_output_stats_path(admin3_id, recipe_ids)
        if ckpt.exists() and not args.reprocess:
            print(f'[{admin3_id}] already done')
            continue

        parcels = load_parcels(admin3_id, recipe_ids)
        if parcels is None:
            print(f'[{admin3_id}] SKIP — no parcels loaded')
            continue

        t0 = time.perf_counter()
        result = run_zonal_stats(parcels, RASTER_PATH)
        elapsed = time.perf_counter() - t0

        if result is None:
            print(f'[{admin3_id}] SKIP — no parcels overlap raster extent')
            continue

        result.to_parquet(ckpt)
        print(f'[{admin3_id}] {len(result):,} parcels  {elapsed:.0f}s  -> {ckpt.name}')

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
COMMIT = False
# If True, writes `.py` script to 'scripts/...'.
# If False, writes a test version to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

In [ ]:
import matplotlib.pyplot as plt

for admin2_id in ADMIN2_IDS:
    admin3_ids = op.get_admin_ids(admin_level=3, admin_id=admin2_id)

    result_frames = []
    parcel_frames = []

    for admin3_id in admin3_ids:
        recipe_ids, _ = find_recipe_group(admin3_id)
        if recipe_ids is None:
            continue
        ckpt = get_output_stats_path(admin3_id, recipe_ids)
        if not ckpt.exists():
            continue
        result = pd.read_parquet(ckpt)
        parcels = load_parcels(admin3_id, recipe_ids)
        if parcels is None:
            continue
        result_frames.append(result)
        parcel_frames.append(parcels.to_crs(RASTER_CRS))

    if not result_frames:
        print(f'{admin2_id}: no checkpoints found')
        continue

    all_results = pd.concat(result_frames)
    all_parcels = gpd.GeoDataFrame(pd.concat(parcel_frames), crs=RASTER_CRS)
    plot_gdf = all_parcels.join(all_results)
    vals = all_results[COL_NAME].dropna()

    valid_gdf = plot_gdf[plot_gdf[COL_NAME].notna()]
    minx, miny, maxx, maxy = valid_gdf.total_bounds
    mx = (maxx - minx) * 0.03
    my = (maxy - miny) * 0.03

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].hist(vals, bins=60, color='steelblue', alpha=0.8)
    axes[0].set_xlabel(COL_NAME)
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{admin2_id}: {COL_NAME} distribution ({METHOD})')
    axes[0].grid(alpha=0.3)

    plot_gdf.plot(
        column=COL_NAME,
        cmap='terrain',
        ax=axes[1],
        legend=True,
        legend_kwds={'shrink': 0.6},
        missing_kwds={'color': 'lightgrey'},
    )
    axes[1].set_xlim(minx - mx, maxx + mx)
    axes[1].set_ylim(miny - my, maxy + my)
    axes[1].set_title(f'{admin2_id}: {COL_NAME}')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
    print(
        f'{admin2_id}: {len(all_results):,} total parcels across {len(result_frames)} municipalities'
    )